# Python Language Review — Session 2
## Whirlwind Tour of Python, chapters 9–14

Topics: Functions · Errors & Exceptions · Iterators · List Comprehensions · Generators · Strings & Regex

Reference: <https://jakevdp.github.io/WhirlwindTourOfPython/>

# WTOP.9 — Defining Functions

Functions are defined with `def`.  Key features:
- **Default argument values** make arguments optional.
- `*args` collects extra positional arguments into a tuple.
- `**kwargs` collects extra keyword arguments into a dict.
- **`lambda`** creates an anonymous single-expression function.
- Functions are **first-class objects** — they can be passed as arguments and returned from other functions.

In [1]:
# Basic def and return
def fibonacci(n, a=0, b=1):
    """Return the first n Fibonacci numbers starting from a, b."""
    result = []
    for _ in range(n):
        result.append(a)
        a, b = b, a + b
    return result

print(fibonacci(8))
print(fibonacci(5, a=2, b=3))   # override defaults

[0, 1, 1, 2, 3, 5, 8, 13]
[2, 3, 5, 8, 13]


In [2]:
# *args and **kwargs
def describe(*args, **kwargs):
    print("Positional:", args)
    print("Keyword   :", kwargs)

describe(1, 2, 3, name="Alice", city="Athens")

Positional: (1, 2, 3)
Keyword   : {'name': 'Alice', 'city': 'Athens'}


In [3]:
# lambda — anonymous function
square  = lambda x: x ** 2
add     = lambda x, y: x + y
print(square(5), add(3, 4))

# Practical: use lambda as a key for sorting
people = [{'name': 'Bob', 'age': 25},
          {'name': 'Alice', 'age': 30},
          {'name': 'Carol', 'age': 22}]

by_age  = sorted(people, key=lambda p: p['age'])
by_name = sorted(people, key=lambda p: p['name'])
print([p['name'] for p in by_age])
print([p['name'] for p in by_name])

25 7
['Carol', 'Bob', 'Alice']
['Alice', 'Bob', 'Carol']


In [4]:
# Functions are first-class objects — can be passed and returned
import random

def gen_password(charset, length):
    """Return a random string drawn from charset of the given length."""
    return ''.join(random.choices(charset, k=length))

def create_password_generator(charset):
    """Return a function that generates passwords from a fixed charset."""
    return lambda length: gen_password(charset, length)

# Build specialised generators from the same factory
alphanumeric = create_password_generator("abcdefghijklmnopqrstuvwxyz0123456789")
numeric_only = create_password_generator("0123456789")

print(alphanumeric(8))
print(numeric_only(6))

m9hbmur7
744186


# WTOP.10 — Errors and Exceptions

Python uses exceptions for error handling.

```
try:
    <risky code>
except <ExceptionType> as e:
    <handle error>
else:
    <runs if no exception>
finally:
    <always runs>
```

Common built-in exceptions: `ValueError`, `TypeError`, `KeyError`, `IndexError`,
`FileNotFoundError`, `ZeroDivisionError`, `AttributeError`.

Use `raise` to throw an exception deliberately.

In [5]:
# try / except
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print("Cannot divide by zero!")
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))

5.0
Cannot divide by zero!
None


In [6]:
# Catch multiple exception types; use else and finally
def parse_int(s):
    try:
        result = int(s)
    except ValueError as e:
        print(f"ValueError: {e}")
        return None
    except TypeError as e:
        print(f"TypeError: {e}")
        return None
    else:
        print("Parsed successfully")
        return result
    finally:
        print("--- parse_int done ---")   # always runs

parse_int("42")
parse_int("hello")
parse_int(None)

Parsed successfully
--- parse_int done ---
ValueError: invalid literal for int() with base 10: 'hello'
--- parse_int done ---
TypeError: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'
--- parse_int done ---


In [7]:
# raise — signal errors from your own code
def set_age(age):
    if not isinstance(age, int):
        raise TypeError(f"age must be int, got {type(age).__name__}")
    if age < 0 or age > 150:
        raise ValueError(f"age {age} is out of realistic range")
    return age

try:
    set_age(-5)
except ValueError as e:
    print("Caught:", e)

try:
    set_age("old")
except TypeError as e:
    print("Caught:", e)

Caught: age -5 is out of realistic range
Caught: age must be int, got str


# WTOP.11 — Iterators

An **iterable** is any object that can be looped over (`list`, `str`, `dict`, files, …).
An **iterator** is an object with a `__next__()` method; you get one by calling `iter()`.

Useful built-ins that return iterators / work with iterables:

| Function | Description |
|----------|-------------|
| `enumerate(it)` | Yields `(index, value)` pairs |
| `zip(a, b)` | Yields `(a_i, b_i)` tuples |
| `map(fn, it)` | Apply `fn` to each element |
| `filter(fn, it)` | Keep elements where `fn` returns True |
| `reversed(seq)` | Iterate in reverse |
| `sorted(it)` | Return a sorted list |

In [8]:
# iter() and next() — under the hood of for loops
nums = [10, 20, 30]
it = iter(nums)
print(next(it))  # 10
print(next(it))  # 20
print(next(it))  # 30
# next(it)       # StopIteration — exhausted

10
20
30


In [9]:
# enumerate, zip
names  = ['Alice', 'Bob', 'Carol']
scores = [92, 87, 95]

for i, (name, score) in enumerate(zip(names, scores)):
    print(f"  {i+1}. {name}: {score}")

  1. Alice: 92
  2. Bob: 87
  3. Carol: 95


In [10]:
# map and filter
numbers = range(1, 11)

squares  = list(map(lambda x: x**2, numbers))
evens    = list(filter(lambda x: x % 2 == 0, numbers))
even_sq  = list(map(lambda x: x**2, filter(lambda x: x % 2 == 0, numbers)))

print("Squares:          ", squares)
print("Evens:            ", evens)
print("Squares of evens: ", even_sq)

Squares:           [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
Evens:             [2, 4, 6, 8, 10]
Squares of evens:  [4, 16, 36, 64, 100]


# WTOP.12 — List Comprehensions

Comprehensions are a compact, readable way to build collections.

```python
[expr for var in iterable]                    # basic
[expr for var in iterable if condition]       # with filter
[expr if condition else alt for var in it]    # conditional value
{expr for var in iterable}                    # set comprehension
{key: val for var in iterable}               # dict comprehension
```

They are generally faster than equivalent `for` + `append` loops.

In [11]:
# Basic list comprehension
squares = [n**2 for n in range(10)]
print(squares)

# Filter: only even squares
even_sq = [n**2 for n in range(10) if n % 2 == 0]
print(even_sq)

# Conditional value: negate odd numbers
result = [n if n % 2 == 0 else -n for n in range(10)]
print(result)

[0, 1, 4, 9, 16, 25, 36, 49, 64, 81]
[0, 4, 16, 36, 64]
[0, -1, 2, -3, 4, -5, 6, -7, 8, -9]


In [12]:
# Nested: Cartesian product
pairs = [(i, j) for i in range(3) for j in range(3) if i != j]
print(pairs)

# Flatten a 2D list
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
flat   = [x for row in matrix for x in row]
print(flat)

[(0, 1), (0, 2), (1, 0), (1, 2), (2, 0), (2, 1)]
[1, 2, 3, 4, 5, 6, 7, 8, 9]


In [13]:
# Set and dict comprehensions
words  = ['apple', 'banana', 'cherry', 'apple', 'banana']
unique = {w for w in words}              # set: removes duplicates
lengths = {w: len(w) for w in unique}   # dict: word -> length
print(unique)
print(lengths)

# Practical: state -> abbreviation mapping
states = ["Alabama", "Alaska", "Arizona"]
abbrev = ["AL", "AK", "AZ"]
state_dict = {s: a for s, a in zip(states, abbrev)}
print(state_dict)

{'apple', 'cherry', 'banana'}
{'apple': 5, 'cherry': 6, 'banana': 6}
{'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ'}


# WTOP.13 — Generators and Generator Expressions

A **generator** produces values one at a time, on demand — it never builds the full list in memory.

- **Generator function**: use `yield` instead of `return`.
- **Generator expression**: like a list comprehension but with `()` — lazy.
- Generators are iterators — consumed once and exhausted.

Generators are ideal for large or infinite sequences.

In [14]:
# Generator function with yield
def count_up(start, stop):
    """Yield integers from start to stop-1."""
    n = start
    while n < stop:
        yield n
        n += 1

gen = count_up(1, 5)
print(next(gen))   # 1
print(next(gen))   # 2
print(list(gen))   # [3, 4] — consume the rest

1
2
[3, 4]


In [15]:
# Infinite generator — would be impossible with a list
def naturals():
    n = 1
    while True:
        yield n
        n += 1

gen = naturals()
first_10 = [next(gen) for _ in range(10)]
print(first_10)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [16]:
# Generator expression (lazy list comprehension)
# Does NOT build the full list in memory
million_squares = (n**2 for n in range(1_000_000))  # created instantly

# Consume only what we need
import itertools
first_five = list(itertools.islice(million_squares, 5))
print(first_five)

# sum() works with generators directly
total = sum(n**2 for n in range(101))
print(total)

[0, 1, 4, 9, 16]
338350


In [17]:
# Generator vs. list: memory comparison
import sys

lst = [n**2 for n in range(10_000)]   # builds full list
gen = (n**2 for n in range(10_000))   # lazy

print(f"List size: {sys.getsizeof(lst):,} bytes")
print(f"Generator size: {sys.getsizeof(gen):,} bytes")

List size: 85,176 bytes
Generator size: 200 bytes


# WTOP.14 — Strings and Regular Expressions

Strings are **immutable** sequences of characters.

**Key string methods:**

| Method | Description |
|--------|-------------|
| `.upper()` / `.lower()` / `.title()` | Case conversion |
| `.strip()` / `.lstrip()` / `.rstrip()` | Remove whitespace |
| `.split(sep)` | Split into a list |
| `sep.join(list)` | Join a list into a string |
| `.find(sub)` / `.replace(old, new)` | Search and replace |
| `.startswith(p)` / `.endswith(p)` | Prefix / suffix check |

**f-strings** (format strings) are the modern way to embed values: `f"Hello, {name}!"`

**Regular expressions** (`import re`) provide flexible pattern matching.

In [18]:
# String methods
s = "  Hello, World!  "
print(s.strip())
print(s.lower())
print(s.title())
print(s.strip().replace("World", "Python"))

# split and join
csv_line = "Alice,30,Athens,Georgia"
parts = csv_line.split(',')
print(parts)
print(" | ".join(parts))

Hello, World!
  hello, world!  
  Hello, World!  
Hello, Python!
['Alice', '30', 'Athens', 'Georgia']
Alice | 30 | Athens | Georgia


In [19]:
# f-strings with format specs
name  = "Bob"
score = 87.5
pi    = 3.14159

print(f"Name: {name:<10} Score: {score:5.1f}")
print(f"Pi ≈ {pi:.3f}")
print(f"Hex: {255:#x}   Binary: {10:#b}")

Name: Bob        Score:  87.5
Pi ≈ 3.142
Hex: 0xff   Binary: 0b1010


In [20]:
import re

# Basic regex patterns
text = "Call us at 706-555-1234 or 800-555-9876 before 5pm."

# Find all phone numbers
phones = re.findall(r'\d{3}-\d{3}-\d{4}', text)
print("Phones:", phones)

# Search: first match
m = re.search(r'(\d{3})-(\d{3}-\d{4})', text)
if m:
    print("Area code:", m.group(1))
    print("Number:   ", m.group(0))

Phones: ['706-555-1234', '800-555-9876']
Area code: 706
Number:    706-555-1234


In [21]:
# re.compile for reuse; groups and named groups
price_pattern = re.compile(r'\$\d+\.\d+|\$\d+')

reviews = [
    "Great product at $29.99!",
    "Not worth $100 at all.",
    "No price mentioned here.",
    "Saw it for $12 and $15 on different sites.",
]

for review in reviews:
    prices = price_pattern.findall(review)
    if prices:
        print(f"  {prices}  <- {review[:40]}")

  ['$29.99']  <- Great product at $29.99!
  ['$100']  <- Not worth $100 at all.
  ['$12', '$15']  <- Saw it for $12 and $15 on different site


In [22]:
# re.sub — find and replace with a pattern
# Redact phone numbers
text = "Reach Alice at 706-555-1234 or Bob at 800-555-9876."
redacted = re.sub(r'\d{3}-\d{3}-\d{4}', '[REDACTED]', text)
print(redacted)

# split on any whitespace or punctuation
sentence = "one,two;three four	five"
tokens = re.split(r'[\s,;]+', sentence)
print(tokens)

Reach Alice at [REDACTED] or Bob at [REDACTED].
['one', 'two', 'three', 'four', 'five']
